In [1]:
from wimby_lca.built_inventory import create_dictionary_update
from wimby_lca.calculations import lca_wimby_fleet_evaluation, lca_wimby_fleet_evaluation_total_impacts
from pathlib import Path
import time
import pandas as pd
import shutil

In [2]:
# -*- coding: utf-8 -*-
"""
End-to-end script to: (1) harden IDs, (2) audit existing outputs, (3) resume safely.
"""
from __future__ import annotations
import hashlib
import glob
import time
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
from pathlib import Path


In [3]:
# STEP 1: Load wind turbine fleet data
wind_turbines = pd.read_csv(r"C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\data\EU_wind_fleet.csv")
wind_turbines.drop(columns=["Unnamed: 0"], inplace=True)
print(wind_turbines.shape)
print(wind_turbines.head())

(77552, 13)
   P_rated_kW  Hub_height_m  Diameter_m  Longitude  Latitude ISO_code  \
0        2300            72         101   -4.34066  55.70955       GB   
1        2300            72         101   -4.32475  55.65482       GB   
2        2300            72         101   -4.30394  55.65528       GB   
3        2300            72         101   -4.33106  55.65598       GB   
4        2300            72         101   -4.31517  55.65733       GB   

   Offshore Commissioning_date  Lifetime_production_kWh  Capacity_factors  \
0         0         01/05/2009               96710400.0              0.24   
1         0         01/05/2009               96710400.0              0.24   
2         0         01/05/2009               96710400.0              0.24   
3         0         01/05/2009               96710400.0              0.24   
4         0         01/05/2009               96710400.0              0.24   

   turbine_age  Cluster  park_size  
0           15     3696        100  
1           

In [4]:
print(len(wind_turbines[wind_turbines['ISO_code']=='DE']))

27916


In [7]:
wind_de_fleet = wind_turbines[wind_turbines['ISO_code']=='DE']
wind_de_fleet['P_rated_kW'].sum()

59945788

In [9]:
#Control of calulated CC impact for DE fleet
cc_de_fleet= pd.read_csv(r"C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\fleet_impacts_DE_MASTER_normalized_kWh.csv")
print(len(cc_de_fleet))
print(cc_de_fleet['P_rated_kW'].sum())

27916
59945788


In [5]:
# =========================
# STEP 2: CONFIG
# =========================
country = "DE"

# --- robust BASE_DIR for scripts and notebooks ---
try:
    BASE_DIR = Path(__file__).resolve().parent  # works in .py scripts
except NameError:
    BASE_DIR = Path.cwd()  # works in Jupyter/IPython

DATA_DIR   = BASE_DIR / "wimby_lca" / "data"
EXCEL_PATH = DATA_DIR / "EU_turbines_input_data.xlsx"

# If you're not running from the project root, auto-find the data file by walking up
if not EXCEL_PATH.exists():
    for parent in [BASE_DIR, *BASE_DIR.parents]:
        candidate = parent / "wimby_lca" / "data" / "EU_turbines_input_data.xlsx"
        if candidate.exists():
            EXCEL_PATH = candidate
            DATA_DIR = candidate.parent
            BASE_DIR = parent
            break

OUT_DIR    = Path(r"C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_PATH  = OUT_DIR / f"fleet_impacts_{country}_MASTER_normalized_kWh.csv"
LEDGER_PATH  = OUT_DIR / f"processed_ids_{country}.csv"
DUPREP_PATH  = OUT_DIR / f"duplicate_turbine_ids_report_{country}.csv"

chunk_size  = 5000
chunk_index = 0  # 0-based

KEY_COLS = ['Longitude', 'Latitude', 'P_rated_kW', 'Hub_height_m', 'Diameter_m', 'park_size', 'Offshore']

print("BASE_DIR:", BASE_DIR)
print("DATA_DIR:", DATA_DIR)
print("EXCEL_PATH exists:", EXCEL_PATH.exists())


BASE_DIR: c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA
DATA_DIR: c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\data
EXCEL_PATH exists: True


In [6]:
# =========================
# Step: 3 ID helpers (Hardened) 12/11/2025
# =========================
def _canon_val(v):
    """Deterministically normalize values before hashing."""
    if pd.isna(v):
        return "NA"
    if isinstance(v, (float, np.floating)):
        return f"{float(v):.6f}"  # 6 decimals ~ 0.1 m at lat/lon; adjust if you need finer
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    return str(v).strip()

def make_turbine_id_sha256(df: pd.DataFrame) -> pd.Series:
    """Create a stable SHA-256 ID from KEY_COLS with rounding & coercion."""
    missing = [c for c in KEY_COLS if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns for ID creation: {missing}")
    # Coerce expected numeric columns
    num_cols = ['Longitude','Latitude','P_rated_kW','Hub_height_m','Diameter_m','park_size','Offshore']
    for c in num_cols:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    def row_to_id(row):
        parts = [_canon_val(row[c]) for c in KEY_COLS]
        payload = "|".join(parts)
        return hashlib.sha256(payload.encode('utf-8')).hexdigest()  # 64 hex chars

    return df.apply(row_to_id, axis=1)

def update_ledger_atomic(ledger_path: Path, new_ids: pd.Series) -> None:
    """Crash-safe ledger write: write to temp then replace."""
    ledger_path.parent.mkdir(parents=True, exist_ok=True)
    if ledger_path.exists():
        ledger = pd.read_csv(ledger_path, dtype=str)
    else:
        ledger = pd.DataFrame(columns=['turbine_id'])
    merged = pd.concat([ledger[['turbine_id']], new_ids.to_frame('turbine_id')], ignore_index=True)\
               .drop_duplicates(subset='turbine_id', keep='first')
    tmp = ledger_path.with_suffix(ledger_path.suffix + ".tmp")
    merged.to_csv(tmp, index=False)
    tmp.replace(ledger_path)

In [7]:
# =========================
#Step 4: 1) Load input & build IDs (12/11/2025)
# =========================
print("Loading input Excel…")
xl = pd.ExcelFile(EXCEL_PATH)
if "EU_turbines_input_data" not in xl.sheet_names:
    raise ValueError(f"Sheet 'EU_turbines_input_data' not found. Available: {xl.sheet_names}")

eu = pd.read_excel(EXCEL_PATH, sheet_name="EU_turbines_input_data")

# Validate expected columns (plus essentials for run)
req_cols = set(KEY_COLS + ['ISO_code', 'Lifetime_production_kWh'])
missing = sorted(list(req_cols - set(eu.columns)))
if missing:
    raise ValueError(f"Input is missing columns: {missing}")

# Filter to country
inp = eu.loc[eu['ISO_code'] == country].copy()

# Normalize Offshore once (0/1 -> bool). This does not affect IDs anymore but keeps downstream logic sane.
inp['Offshore'] = pd.to_numeric(inp['Offshore'], errors='coerce').fillna(0).astype(int).astype(bool)

# NEW: Build sequential per-country IDs (DE_00000 ... DE_27915)
#     This guarantees a unique ID per row, regardless of layout.
inp = inp.reset_index(drop=True)  # stable within current file ordering
inp['turbine_id'] = [f"{country}_{i:05d}" for i in range(len(inp))]  # NEW

print(f"Rows in input for {country}: {len(inp)}")
print(f"First/last IDs: {inp['turbine_id'].iloc[0]} … {inp['turbine_id'].iloc[-1]}")

Loading input Excel…
Rows in input for DE: 27916
First/last IDs: DE_00000 … DE_27915


In [9]:
# =========================
# STEP 5: 
# Updated: 12/11/2025
#  2) Audit existing outputs and seed MASTER + LEDGER
#    (now requires turbine_id to be present; do NOT rebuild IDs)
# =========================
print("Auditing existing chunk outputs…")
pattern = str(OUT_DIR / f"fleet_impacts_{country}*_normalized_kWh.csv")
files = [f for f in glob.glob(pattern)
         if "MASTER" not in f and "RESUME" not in f]  # ignore master & resume chunks
files = sorted(files)  # deterministic

dfs: List[pd.DataFrame] = []
for fp in files:
    df = pd.read_csv(fp, low_memory=False)
    if 'turbine_id' not in df.columns:
        # CHANGED: error on missing ID; we rely on sequential IDs now
        raise ValueError(f"{Path(fp).name} is missing turbine_id; cannot audit with sequential scheme.")  # CHANGED
    df['turbine_id'] = df['turbine_id'].astype(str)
    df['__source_file'] = Path(fp).name
    dfs.append(df)

if dfs:
    all_out = pd.concat(dfs, ignore_index=True)
    dup_mask = all_out.duplicated('turbine_id', keep=False)
    n_dups = int(dup_mask.sum())
    print(f"Found {len(all_out)} rows across {len(files)} files; duplicates across files: {n_dups}")

    if n_dups:
        dup_report = (
            all_out.loc[dup_mask, ['turbine_id','__source_file']]
                   .drop_duplicates()
                   .sort_values(['turbine_id','__source_file'])
                   .groupby('turbine_id')['__source_file'].apply(list)
                   .reset_index().rename(columns={'__source_file':'files'})
        )
        dup_report['files'] = dup_report['files'].apply(lambda L: ';'.join(L))
        dup_report.to_csv(DUPREP_PATH, index=False)
        print(f"Duplicate report written: {DUPREP_PATH}")

    # Deduped master (keep first occurrence deterministically by filename order)
    master = (
        all_out.sort_values(['turbine_id','__source_file'])
               .drop_duplicates(subset='turbine_id', keep='first')
    )
    master.to_csv(MASTER_PATH, index=False)
    print(f"MASTER written: {MASTER_PATH} with {len(master)} unique turbines")

    # Seed ledger from master
    def update_ledger_atomic(ledger_path: Path, new_ids: pd.Series) -> None:
        ledger_path.parent.mkdir(parents=True, exist_ok=True)
        if ledger_path.exists():
            ledger = pd.read_csv(ledger_path, dtype=str)
        else:
            ledger = pd.DataFrame(columns=['turbine_id'])
        merged = pd.concat([ledger[['turbine_id']], new_ids.to_frame('turbine_id')], ignore_index=True)\
                   .drop_duplicates(subset='turbine_id', keep='first')
        tmp = ledger_path.with_suffix(ledger_path.suffix + ".tmp")
        merged.to_csv(tmp, index=False)
        tmp.replace(ledger_path)

    update_ledger_atomic(LEDGER_PATH, master['turbine_id'].astype(str))
    print(f"Ledger seeded: {LEDGER_PATH}")
else:
    print("No precomputed chunk files found to audit.")
    # If no prior outputs, ensure an empty MASTER/LEDGER exist
    if not MASTER_PATH.exists():
        pd.DataFrame(columns=['turbine_id']).to_csv(MASTER_PATH, index=False)
    if not LEDGER_PATH.exists():
        pd.DataFrame(columns=['turbine_id']).to_csv(LEDGER_PATH, index=False)

# Reload ledger after seeding (for next step)
ledger = pd.read_csv(LEDGER_PATH, dtype=str)['turbine_id'].tolist() if LEDGER_PATH.exists() else []


Auditing existing chunk outputs…
Found 24060 rows across 6 files; duplicates across files: 2288
Duplicate report written: C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\duplicate_turbine_ids_report_DE.csv
MASTER written: C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\fleet_impacts_DE_MASTER_normalized_kWh.csv with 22916 unique turbines
Ledger seeded: C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\processed_ids_DE.csv


In [19]:
(len(wind_turbines[wind_turbines['ISO_code']=='DE']))-(24060-2288)

6144

In [23]:
(len(wind_turbines[wind_turbines['ISO_code']=='DE']))-22916

5000

In [22]:
24060-22916

1144

In [21]:
24060+2288

26348

In [10]:
# =========================
# STEP 6:
#  Updated: 12/11/2025
# 3) Determine remaining, chunk deterministically, run LCA safely
# =========================
remaining = inp[~inp['turbine_id'].isin(ledger)].copy()
# Deterministic order by ID (already sequential, but keep it explicit)
remaining = remaining.sort_values('turbine_id').reset_index(drop=True)
print(f"Remaining to compute for {country}: {remaining.shape[0]}")

if remaining.empty:
    print("Nothing left to compute. You’re fully up to date.")
else:
    chunks = [remaining.iloc[i:i+chunk_size] for i in range(0, len(remaining), chunk_size)]
    if chunk_index >= len(chunks):
        raise SystemExit(f"chunk_index {chunk_index} out of range. There are only {len(chunks)} chunks.")
    country_chunk = chunks[chunk_index].copy()
    print(f"Running chunk {chunk_index+1}/{len(chunks)} with {country_chunk.shape[0]} turbines")

    # Ensure lifecycle columns exist
    lifecycle_stages_onshore  = ['Input','Assembly','Transport','Maintenance','Disposal','Total']
    lifecycle_stages_offshore = ['Input','Assembly','Transport','Disposal','Total']
    for col in set(lifecycle_stages_onshore + lifecycle_stages_offshore):
        if col not in country_chunk.columns:
            country_chunk[col] = None

    # --- Your LCA functions ---
    from wimby_lca.built_inventory import create_dictionary_update
    from wimby_lca.calculations import lca_wimby_fleet_evaluation

    climate_change = ('EF v3.1', 'climate change', 'global warming potential (GWP100)')

    # Crash-safe periodic ledger update util (reuse above)
    def update_ledger_atomic(ledger_path: Path, new_ids: pd.Series) -> None:
        ledger_path.parent.mkdir(parents=True, exist_ok=True)
        if ledger_path.exists():
            ledger = pd.read_csv(ledger_path, dtype=str)
        else:
            ledger = pd.DataFrame(columns=['turbine_id'])
        merged = pd.concat([ledger[['turbine_id']], new_ids.to_frame('turbine_id')], ignore_index=True)\
                   .drop_duplicates(subset='turbine_id', keep='first')
        tmp = ledger_path.with_suffix(ledger_path.suffix + ".tmp")
        merged.to_csv(tmp, index=False)
        tmp.replace(ledger_path)

    # Run
    start_time = time.time()
    processed_ids_buffer = []  # for periodic ledger flush (crash-safety)
    FLUSH_EVERY = 250          # update ledger every N rows

    for i, (idx, row) in enumerate(country_chunk.iterrows(), start=1):
        try:
            dct = create_dictionary_update(
                P=row['P_rated_kW'],
                lon=row['Longitude'],
                lat=row['Latitude'],
                h=row['Hub_height_m'],
                d=row['Diameter_m'],
                park_size=row['park_size'],
                print_details=False
            )
            res = lca_wimby_fleet_evaluation(
                dict_activities=dct,
                impact_category=climate_change,
                aep=row['Lifetime_production_kWh']
            )
            if res is None:
                res = {s: [0] for s in lifecycle_stages_onshore}

            stages = lifecycle_stages_offshore if row['Offshore'] else lifecycle_stages_onshore
            for s in stages:
                country_chunk.at[idx, s] = res.get(s, [0])[0]

        except Exception as e:
            print(f"[WARN] turbine_id {row['turbine_id']}: {e}")
            for s in set(lifecycle_stages_onshore + lifecycle_stages_offshore):
                country_chunk.at[idx, s] = 0

        # Crash-safe periodic ledger update
        processed_ids_buffer.append(str(row['turbine_id']))
        if len(processed_ids_buffer) >= FLUSH_EVERY:
            update_ledger_atomic(LEDGER_PATH, pd.Series(processed_ids_buffer, name='turbine_id'))
            processed_ids_buffer.clear()

    # Final flush
    if processed_ids_buffer:
        update_ledger_atomic(LEDGER_PATH, pd.Series(processed_ids_buffer, name='turbine_id'))

    # Save the computed chunk (ALWAYS keep turbine_id)
    chunk_out = OUT_DIR / f"fleet_impacts_{country}_RESUME_chunk{chunk_index+1}_normalized_kWh.csv"
    country_chunk.to_csv(chunk_out, index=False)
    print(f"Saved chunk: {chunk_out}")

    # Update master (deduped by turbine_id just in case)
    if MASTER_PATH.exists() and MASTER_PATH.stat().st_size > 0:
        master_now = pd.read_csv(MASTER_PATH, low_memory=False)
        master_now = pd.concat([master_now, country_chunk], ignore_index=True)\
                       .drop_duplicates(subset='turbine_id', keep='first')
    else:
        master_now = country_chunk.copy()
    master_now.to_csv(MASTER_PATH, index=False)
    print(f"MASTER updated: {MASTER_PATH} with {len(master_now)} unique turbines "
          f"(chunk time {time.time()-start_time:.1f}s)")

Remaining to compute for DE: 5000
Running chunk 1/1 with 5000 turbines


c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_act = pd.concat([df_act, df_act2], ignore_index=True)
c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:211: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_act = df_act.replace('ecoinvent cutoff 391', eidb.name)


Existing MV transformer dataset found.
Existing HV transformer dataset found.
Your dataset was successfully built or retrieved!


c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_act = pd.concat([df_act, df_act2], ignore_index=True)
c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:211: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_act = df_act.replace('ecoinvent cutoff 391', eidb.name)


Existing MV transformer dataset found.
Existing HV transformer dataset found.
Your dataset was successfully built or retrieved!


...

Existing MV transformer dataset found.
Existing HV transformer dataset found.
Your dataset was successfully built or retrieved!


c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_act = pd.concat([df_act, df_act2], ignore_index=True)
c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:211: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_act = df_act.replace('ecoinvent cutoff 391', eidb.name)


Existing MV transformer dataset found.
Existing HV transformer dataset found.
Your dataset was successfully built or retrieved!


c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:195: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_act = pd.concat([df_act, df_act2], ignore_index=True)
c:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Github_repo\WIMBY_LCA\wimby_lca\prepare_inventories.py:211: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_act = df_act.replace('ecoinvent cutoff 391', eidb.name)


Existing MV transformer dataset found.
Existing HV transformer dataset found.
Your dataset was successfully built or retrieved!
Saved chunk: C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\fleet_impacts_DE_RESUME_chunk1_normalized_kWh.csv
MASTER updated: C:\Users\dhuber\OneDrive - Vrije Universiteit Brussel\DominikMaeva\01_Projects\05_WIMBY\05_LCA\Evaulation\Fleet_results\Fleet_results_DE\fleet_impacts_DE_MASTER_normalized_kWh.csv with 27916 unique turbines (chunk time 286172.4s)
